# Seven-Arm Dependency-Aware Selective Regeneration Benchmark

**SMOKE / NON-PUBLICATION**

Runs the benchmark entirely from attached Kaggle Datasets. No GitHub clone.

- **Smoke profile** (default): 1 scenario, 7 strategies, non-publication.
- **Pilot profile** (requires changing `--profile smoke` to `--profile pilot`): 12 scenarios, 2 strategies, 2 reps.
- **Research profile** (requires changing `--profile smoke` to `--profile research`): 24 scenarios, 4 strategies, 3 reps.

Only smoke runs automatically. Pilot and research require manual profile change.

In [ ]:
import os
import sys
from pathlib import Path

# ---- Discover Kaggle Datasets ----------------------------------------------
KAGGLE_INPUT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/runs")

KNOWN_CODE = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-code"
KNOWN_DATA = KAGGLE_INPUT / "datasets/ahmedehabh/dependency-aware-selective-regeneration-data"
KNOWN_MODEL = KAGGLE_INPUT / "models/qwen-lm/qwen2.5-coder/transformers/7b-instruct/1"
FALLBACK_CODE = KAGGLE_INPUT / "dependency-aware-selective-regeneration-code"
FALLBACK_DATA = KAGGLE_INPUT / "dependency-aware-selective-regeneration-data"

def discover(label, candidates, required_subdir=None):
    for p in candidates:
        if p.is_dir():
            if required_subdir is None or (p / required_subdir).is_dir():
                return p
            print(f"  [info] {p.name} exists but missing '{required_subdir}'")
    if KAGGLE_INPUT.is_dir():
        for entry in sorted(KAGGLE_INPUT.iterdir()):
            if entry.is_dir() and (required_subdir is None or (entry / required_subdir).is_dir()):
                return entry
    raise FileNotFoundError(f"Cannot find {label} in {KAGGLE_INPUT}")

CODE_DIR = discover("code dataset", [KNOWN_CODE, FALLBACK_CODE], required_subdir="src")
DATA_DIR = discover("data dataset", [KNOWN_DATA, FALLBACK_DATA], required_subdir="scenarios")

src_dir = CODE_DIR / "src"
if src_dir.is_dir():
    sys.path.insert(0, str(src_dir))
else:
    raise FileNotFoundError(f"src/ not found in code dataset: {CODE_DIR}")

if KNOWN_MODEL.is_dir():
    MODEL_PATH = str(KNOWN_MODEL.resolve())
else:
    MODEL_PATH = ""
    print("WARNING: Qwen model not found at known path")

SCRIPT_PATH = CODE_DIR / "seven_arm_benchmark.py"
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(f"seven_arm_benchmark.py not found in {CODE_DIR}")

SOURCE_TAG = "v0.7.0-smoke-passed"
HF_RESULTS_REPO_ID = "NabilDo/selective-regeneration-experiment-results"

# ---- Read HF_TOKEN from Kaggle Secrets -----------------------------------
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
who = api.whoami()
info = api.repo_info(HF_RESULTS_REPO_ID, repo_type="dataset")
if not info.private:
    raise RuntimeError(f"Repository '{HF_RESULTS_REPO_ID}' is NOT private")

print(f"Code dataset:  {CODE_DIR}")
print(f"Data dataset:  {DATA_DIR}")
print(f"Qwen model:    {MODEL_PATH}")
print(f"Script:        {SCRIPT_PATH}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"Source tag:    {SOURCE_TAG}")
print(f"HF repo:       {HF_RESULTS_REPO_ID}")
print(f"HF auth as:    {who['name']}")
print(f"HF repo:       PRIVATE -- OK")
print("\nSetup complete. All variables defined. Safe to rerun.")

## Auto-Resume Behavior

This cell runs the benchmark with `--auto-resume-hf` which:

1. **Searches** Hugging Face for compatible incomplete experiments under the canonical prefix:
   `experiments/{profile}/{protocol_version}/{source_commit}/`

2. **Discovers candidates** by downloading each experiment's `checkpoint.json` and `run_records.jsonl`

3. **Validates compatibility** against current run:
   - Profile (smoke/pilot/research)
   - Protocol version
   - Source commit (tag/commit)
   - Config hash
   - Model identity
   - Scenario IDs
   - Strategy names

4. **Selects action**:
   - **RESUME** if exactly one compatible *incomplete* experiment found → skips completed runs, continues from next
   - **ALREADY_COMPLETE** if one compatible *complete* experiment found → exits
   - **START_NEW** if no compatible experiment found → creates new experiment
   - **ERROR** if multiple compatible *incomplete* experiments found → requires `--experiment-id`

5. **Every candidate and rejection reason is logged at INFO level** with full diagnostic details

**Key points**:
- `START_NEW` is acceptable only when no compatible candidate exists
- If an existing experiment is rejected unexpectedly, STOP execution and investigate
- All rejection reasons are logged at INFO level with full diagnostic detail
- The canonical Run ID flows unchanged through all artifacts (checkpoint, records, summaries)

In [ ]:
import subprocess

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV["PYTHONPATH"] = str(CODE_DIR / "src") + (
    os.pathsep + SUBPROCESS_ENV["PYTHONPATH"] if SUBPROCESS_ENV.get("PYTHONPATH") else ""
)

exec_cmd = [
    sys.executable, str(SCRIPT_PATH),
    "--profile", "smoke",
    "--max-runs", "1",
    "--hf-sync",
    "--auto-resume-hf",
    "--hf-repo-id", HF_RESULTS_REPO_ID,
    "--source-tag", SOURCE_TAG,
    "--data-dir", str(DATA_DIR),
    "--model-path", MODEL_PATH,
    "--output-dir", str(OUTPUT_DIR),
]
print("Running:", " ".join(exec_cmd))
print("\n--- Output ---")
result = subprocess.run(exec_cmd, capture_output=True, text=True, env=SUBPROCESS_ENV)
print(result.stdout)
if result.stderr:
    print("--- STDERR ---")
    print(result.stderr)
print(f"\nReturn code: {result.returncode}")
if result.returncode != 0:
    raise RuntimeError(f"Benchmark failed with return code {result.returncode}")

In [ ]:
import json
from pathlib import Path

run_dir = OUTPUT_DIR
print(f"Output directory: {run_dir}\n")

# Experiment ID
exp_id_path = run_dir / "experiment_id.txt"
if exp_id_path.exists():
    exp_id = exp_id_path.read_text().strip()
    print(f"Experiment ID: {exp_id}")
else:
    print("Experiment ID: (not found)")

# Checkpoint
cp_path = run_dir / "checkpoint.json"
if cp_path.exists():
    cp = json.loads(cp_path.read_text())
    total = cp.get("total_planned", 0)
    completed = len(cp.get("completed_run_ids", []))
    failed = len(cp.get("failed_run_ids", []))
    pending = len(cp.get("pending_run_ids", []))
    print(f"\nCheckpoint:")
    print(f"  Total:     {total}")
    print(f"  Completed: {completed}")
    print(f"  Failed:    {failed}")
    print(f"  Pending:   {pending}")
    print(f"  Status:    {cp.get('completion_status', 'unknown')}")
else:
    print("Checkpoint: (not found)")

# Run records
records_path = run_dir / "run_records.jsonl"
if records_path.exists():
    lines = [l for l in records_path.read_text().strip().split("\n") if l]
    print(f"\nRun records ({len(lines)} total):")
    completed_ids = []
    pending_ids = []
    for line in lines:
        rec = json.loads(line)
        rid = rec.get("run_id", "?")
        status = rec.get("status", "?")
        marker = "  " if status in ("succeeded", "failed", "timed_out") else "  "
        print(f"{marker}{rid}: {status}")
        if status in ("succeeded", "failed", "timed_out", "cancelled"):
            completed_ids.append(rid)
        else:
            pending_ids.append(rid)
else:
    print("Run records: (not found)")
    completed_ids = []
    pending_ids = []

# HF sync state
sync_path = run_dir / "remote_sync.json"
if sync_path.exists():
    sync = json.loads(sync_path.read_text())
    print(f"\nHF sync:")
    print(f"  Last sync:   {sync.get('last_sync', 'unknown')}")
    print(f"  Remote path: {sync.get('remote_path', 'unknown')}")
    print(f"  Timestamp:   {sync.get('timestamp', 'unknown')}")
else:
    print("\nHF sync: (not found)")

# Next pending run
if pending_ids:
    print(f"\nNext pending run: {pending_ids[0]}")
else:
    print("\nAll runs complete.")

## Notes

- **Smoke**: default profile (1 scenario x 7 strategies, non-publication).
- **Pilot**: change `--profile smoke` to `--profile pilot` (12 scenarios, agent + selective, 2 reps).
- **Research**: change `--profile smoke` to `--profile research` (24 scenarios, 4 strategies, 3 reps).
- All outputs go to `/kaggle/working/runs/`.
- Internet is required for Hugging Face result synchronization.
- `HF_TOKEN` is required and read from Kaggle Secrets.
- Qwen model loading remains offline from the attached Kaggle Model.
- To start a new experiment intentionally, add `--new-experiment` to the command in the execution cell.